# C3-gradient-descent — Practice p19

**Type:** scenario analysis · **Difficulty:** advanced · **Concepts:** gradient-descent, learning-rate, mse-loss

*Reasoning is required. Coding is limited to committing five typed diagnosis strings and then running the marked verifier.*

Five teams intended to run mean-MSE gradient descent. Each transcript contains exactly one planted bug. Run the transcript cell unchanged, diagnose from its printed trace, and write all five diagnoses on paper **before** editing the answer cell. Use each canonical diagnosis exactly once:

- `"un-zeroed-accumulator"`
- `"eta-too-large"`
- `"wrong-axis-broadcast"`
- `"gradient-sign-flip"`
- `"sum-vs-mean"`

The fields are: losses after successive steps; gradient norm; $\nabla L\cdot\Delta\theta$ (negative for a descent-direction update); `step_over_eta_grad`, the applied step length divided by $\eta\lVert\nabla L\rVert$, which is $1.0$ whenever the update uses the current gradient alone; observed and intended residual shapes; and, for a duplicated-batch probe, the number of copies and the measured gradient-norm ratio. Two transcripts combine descent-direction updates (`step_dot_grad` negative at every step) with a gradient norm that grows at every step; only `step_over_eta_grad` separates those two, so cite it explicitly for them. Explain each diagnosis in one sentence citing a trace field, then commit the exact typed identifiers `diagnosis_a` through `diagnosis_e`.

**Banned (zero points): editing the transcript artifact, running or reading the verifier before all five diagnoses are committed, assigning any expected/verdict value yourself, or replacing the required named variables with one anonymous list.**

In [ ]:
transcripts = [
    {"name": "A", "loss": [2.76, 3.04, 3.37, 3.75, 4.19],
     "grad_norm": [2.8, 3.0, 3.2, 3.4, 3.7],
     "step_dot_grad": [0.39, 0.45, 0.51, 0.58, 0.68],
     "step_over_eta_grad": [1.0, 1.0, 1.0, 1.0, 1.0],
     "residual_shape": (12,), "expected_residual_shape": (12,),
     "copies": 1, "grad_ratio": 1.0},
    {"name": "B", "loss": [2.24, 1.71, 1.53, 1.49, 1.48],
     "grad_norm": [2.2, 1.5, 1.0, 0.7, 0.5],
     "step_dot_grad": [-0.24, -0.12, -0.06, -0.03, -0.02],
     "step_over_eta_grad": [1.0, 1.0, 1.0, 1.0, 1.0],
     "residual_shape": (12, 12), "expected_residual_shape": (12,),
     "copies": 1, "grad_ratio": 1.0},
    {"name": "C", "loss": [3.84, 9.27, 24.61, 69.84, 205.17],
     "grad_norm": [4.1, 7.2, 12.5, 22.0, 40.3],
     "step_dot_grad": [-0.81, -2.49, -7.66, -23.8, -74.2],
     "step_over_eta_grad": [1.0, 1.0, 1.0, 1.0, 1.0],
     "residual_shape": (12,), "expected_residual_shape": (12,),
     "copies": 1, "grad_ratio": 1.0},
    {"name": "D", "loss": [6.31, 4.82, 3.15, 2.94, 5.88],
     "grad_norm": [3.2, 5.9, 8.7, 11.8, 15.2],
     "step_dot_grad": [-0.51, -2.68, -7.74, -17.46, -34.05],
     "step_over_eta_grad": [1.0, 1.54, 2.05, 2.51, 2.95],
     "residual_shape": (12,), "expected_residual_shape": (12,),
     "copies": 1, "grad_ratio": 1.0},
    {"name": "E", "loss": [5.12, 4.43, 3.91, 3.56, 3.34],
     "grad_norm": [18.6, 15.1, 12.4, 10.3, 8.7],
     "step_dot_grad": [-1.73, -1.14, -0.77, -0.53, -0.38],
     "step_over_eta_grad": [1.0, 1.0, 1.0, 1.0, 1.0],
     "residual_shape": (12,), "expected_residual_shape": (12,),
     "copies": 5, "grad_ratio": 5.0},
]

for trace in transcripts:
    print(trace)


Write one evidence sentence for each transcript here before committing the diagnosis strings below.

- **A:**
- **B:**
- **C:**
- **D:**
- **E:**

In [ ]:
diagnosis_a: str = ""
diagnosis_b: str = ""
diagnosis_c: str = ""
diagnosis_d: str = ""
diagnosis_e: str = ""

committed_diagnoses: list[str] = [
    diagnosis_a, diagnosis_b, diagnosis_c, diagnosis_d, diagnosis_e
]


### MARKED VERIFICATION CELL

Run this only after all five named answers are committed. It derives the grading codes from the actual `transcripts` artifact and reports agreement only; it never prints expected diagnoses.

In [ ]:
import hashlib
import json

import numpy as np

TRANSCRIPT_DIGEST = "012114ff7a167c7910f6623c8e3f50f36b54cfc0017f40d1766bc804234cf79f"
if hashlib.sha256(json.dumps(transcripts, sort_keys=True).encode()).hexdigest() != TRANSCRIPT_DIGEST:
    raise RuntimeError("the transcript artifact has been modified; restore the cell and re-run")

expected_count = len(transcripts)
if len(committed_diagnoses) != expected_count:
    raise RuntimeError(
        f"committed collection has length {len(committed_diagnoses)}; expected {expected_count}"
    )
if any(not isinstance(value, str) or not value.strip() for value in committed_diagnoses):
    raise RuntimeError("all five named diagnosis strings must be committed before verification")

diagnosis_codes = {
    "eta-too-large": 0,
    "sum-vs-mean": 1,
    "gradient-sign-flip": 2,
    "un-zeroed-accumulator": 3,
    "wrong-axis-broadcast": 4,
}
if any(value not in diagnosis_codes for value in committed_diagnoses):
    raise RuntimeError("each committed diagnosis must use one of the five canonical strings")
if len(set(committed_diagnoses)) != expected_count:
    raise RuntimeError("each canonical diagnosis must be used exactly once")


def fault_signatures(trace):
    """Every signature this trace exhibits. A well-formed transcript exhibits exactly one,
    so the grade never depends on the order these tests are written in."""
    # Non-degeneracy precondition. Several signatures are comparisons over ADJACENT steps, and
    # those are vacuously true for a one-step trace: a single-step transcript would silently
    # match the accumulator signature and nothing else. Require at least two steps and equal
    # field lengths before any signature is evaluated (gate finding, plan 014).
    series = [trace["loss"], trace["grad_norm"], trace["step_dot_grad"], trace["step_over_eta_grad"]]
    if len({len(values) for values in series}) != 1 or len(series[0]) < 2:
        raise RuntimeError(
            f"transcript {trace['name']} is degenerate: every trace series must have the same "
            "length and cover at least two steps"
        )
    ratios = trace["step_over_eta_grad"]
    current_gradient_only = all(np.isclose(r, 1.0, atol=1e-9, rtol=0) for r in ratios)
    hits = set()
    if tuple(trace["residual_shape"]) != tuple(trace["expected_residual_shape"]):
        hits.add(4)
    if all(value > 0 for value in trace["step_dot_grad"]):
        hits.add(2)
    if trace["copies"] > 1 and np.isclose(
        trace["grad_ratio"], float(trace["copies"]), atol=1e-12, rtol=0
    ):
        hits.add(1)
    if all(b > a for a, b in zip(ratios, ratios[1:])):
        hits.add(3)
    if current_gradient_only and trace["loss"][-1] > 20 * trace["loss"][0]:
        hits.add(0)
    return hits


def derive_fault_code(trace):
    hits = fault_signatures(trace)
    if len(hits) != 1:
        raise RuntimeError(
            f"transcript {trace['name']} matches {len(hits)} signatures; each must match exactly one"
        )
    return hits.pop()


derived_codes = [derive_fault_code(trace) for trace in transcripts]
committed_codes = [diagnosis_codes[value] for value in committed_diagnoses]
agreement = [hand == derived for hand, derived in zip(committed_codes, derived_codes)]
print("all_agree:", all(agreement))
